In [2]:
# Import necessary libraries
import numpy as np  # For numerical operations and array handling
import xarray as xr  # For working with labeled multi-dimensional arrays (netCDF files)

# Load the array of considered cell indices/masks
considered_cells = np.load("/Net/Groups/BGI/scratch/fhuang/utrack/Dataset/Further data/considered_cells.npy")

# Load surface area per cell data
sa_x_cell = np.load("sa_x_cell.npy")

# Load and open climate variable datasets (deseasonalized and detrended)
# Sea Surface Temperature data
sst = xr.open_dataarray("/Net/Groups/BGI/scratch/fhuang/sm_uta_cnn/input_data/deseason_detrend/sst.nc")

# Soil Moisture data
sm = xr.open_dataarray("/Net/Groups/BGI/scratch/fhuang/sm_uta_cnn/input_data/deseason_detrend/sm.nc")

# Leaf Area Index data
lai = xr.open_dataarray("/Net/Groups/BGI/scratch/fhuang/sm_uta_cnn/input_data/deseason_detrend/lai.nc")

# 2m Temperature data
t2m = xr.open_dataarray("/Net/Groups/BGI/scratch/fhuang/sm_uta_cnn/input_data/deseason_detrend/t2m.nc")

# Surface Solar Radiation Downwards data
ssrd = xr.open_dataarray("/Net/Groups/BGI/scratch/fhuang/sm_uta_cnn/input_data/deseason_detrend/ssrd.nc")

# Load and open static soil property datasets
# Bulk density of the fine earth fraction
bdod = xr.open_dataarray("/Net/Groups/BGI/scratch/fhuang/sm_uta_cnn/input_data/static/bdod.nc")

# Sand content fraction
sand = xr.open_dataarray("/Net/Groups/BGI/scratch/fhuang/sm_uta_cnn/input_data/static/sand.nc")

# Silt content fraction
silt = xr.open_dataarray("/Net/Groups/BGI/scratch/fhuang/sm_uta_cnn/input_data/static/silt.nc")

# Clay content fraction
clay = xr.open_dataarray("/Net/Groups/BGI/scratch/fhuang/sm_uta_cnn/input_data/static/clay.nc")

# Load precipitation data (total precipitation)
tp = xr.open_dataarray("/Net/Groups/BGI/scratch/fhuang/lac_uta_res/input_data2/tp2.nc")

In [3]:
# Import function to get latitude and longitude coordinates
from get_lat_lon import get_lat_lon

# Get number of samples from the surface area data
nsample = sa_x_cell.shape[0]

# Define the window size for the neighborhood (10 cells in each direction)
inv = 10

# Initialize input data array with NaN values
# Dimensions: [samples, time_steps(216), window_height(21), window_width(21), features(10)]
input_data = np.full((nsample, 216, inv*2+1, inv*2+1, 10), np.nan)

# Initialize target soil moisture data array with NaN values
# Dimensions: [samples, time_steps(216)]
smi_data = np.full((nsample, 216), np.nan)

# Initialize counter and lists for tracking NaN cells
k = 0
x_cell_nan = []
x_cell_nan_idx = []

# Loop through each sample/cell
for i in range(nsample):
    # Get current cell index
    x_cell = sa_x_cell[i]
    
    # Get latitude and longitude indices for the current cell
    lati, loni = considered_cells[:, x_cell]
    
    # Adjust longitude index (shift by 120 degrees)
    loni = loni - 120
    
    # Get actual latitude and longitude coordinates
    lat, lon = get_lat_lon(x_cell)
    
    # Adjust latitude index (convert to 0-based indexing)
    lati = lati - 1
    
    # Calculate the boundaries of the neighborhood window
    up = lati - inv      # Top boundary
    down = lati + inv + 1  # Bottom boundary
    left = loni - inv    # Left boundary
    right = loni + inv + 1  # Right boundary
    
    # Extract data for each variable within the neighborhood window
    
    # Time-varying variables (with temporal dimension)
    laiv = lai.data[:, up:down, left:right]    # Leaf Area Index
    sstv = sst.data[:, up:down, left:right]    # Sea Surface Temperature
    t2mv = t2m.data[:, up:down, left:right]    # 2m Temperature
    rnv = ssrd.data[:, up:down, left:right]    # Surface Solar Radiation
    
    # Static variables (no temporal dimension)
    bdodv = bdod.data[up:down, left:right]     # Bulk density
    sandv = sand.data[up:down, left:right]     # Sand content
    siltv = silt.data[up:down, left:right]     # Silt content
    clayv = clay.data[up:down, left:right]     # Clay content
    
    # Time-varying variables continued
    tpv = tp.data[:, up:down, left:right]      # Total precipitation
    smv = sm.data[:, up:down, left:right]      # Soil moisture (neighborhood)
    
    # Extract soil moisture at the center cell only (target variable)
    smv_i = sm.data[:, lati, loni]
    
    # Store all extracted data in the input array
    input_data[i, :, :, :, 0] = laiv      # Feature 0: LAI
    input_data[i, :, :, :, 1] = sstv      # Feature 1: SST
    input_data[i, :, :, :, 2] = t2mv      # Feature 2: Temperature
    input_data[i, :, :, :, 3] = rnv       # Feature 3: Solar radiation
    
    # Static features (repeated across time dimension)
    input_data[i, :, :, :, 4] = bdodv     # Feature 4: Bulk density
    input_data[i, :, :, :, 5] = sandv     # Feature 5: Sand content
    input_data[i, :, :, :, 6] = siltv     # Feature 6: Silt content
    input_data[i, :, :, :, 7] = clayv     # Feature 7: Clay content
    
    # Time-varying features continued
    input_data[i, :, :, :, 8] = tpv       # Feature 8: Precipitation
    input_data[i, :, :, :, 9] = smv       # Feature 9: Soil moisture (neighborhood)
    
    # Store target variable (center cell soil moisture time series)
    smi_data[i, :] = smv_i
    
    # Print progress information
    print(i, x_cell, i, lati, loni, lat, lon)
    

# Save the prepared input and target data to files
np.save('/Net/Groups/BGI/scratch/fhuang/sm_uta_cnn/inv-10/x.npy', input_data)  # Input features
np.save('/Net/Groups/BGI/scratch/fhuang/sm_uta_cnn/inv-10/y.npy', smi_data)    # Target values

0 5920 0 43 67 13.5 -79.5
1 5921 1 43 79 13.5 -61.5
2 5922 2 43 80 13.5 -60.0
3 6004 3 44 71 12.0 -73.5
4 6005 4 44 72 12.0 -72.0
5 6006 5 44 73 12.0 -70.5
6 6007 6 44 74 12.0 -69.0
7 6008 7 44 75 12.0 -67.5
8 6009 8 44 76 12.0 -66.0
9 6010 9 44 77 12.0 -64.5
10 6011 10 44 78 12.0 -63.0
11 6012 11 44 79 12.0 -61.5
12 6013 12 44 80 12.0 -60.0
13 6094 13 45 69 10.5 -76.5
14 6095 14 45 70 10.5 -75.0
15 6096 15 45 71 10.5 -73.5
16 6097 16 45 72 10.5 -72.0
17 6098 17 45 73 10.5 -70.5
18 6099 18 45 74 10.5 -69.0
19 6100 19 45 75 10.5 -67.5
20 6101 20 45 76 10.5 -66.0
21 6102 21 45 77 10.5 -64.5
22 6103 22 45 78 10.5 -63.0
23 6104 23 45 79 10.5 -61.5
24 6105 24 45 80 10.5 -60.0
25 6187 25 46 67 9.0 -79.5
26 6188 26 46 68 9.0 -78.0
27 6189 27 46 69 9.0 -76.5
28 6190 28 46 70 9.0 -75.0
29 6191 29 46 71 9.0 -73.5
30 6192 30 46 72 9.0 -72.0
31 6193 31 46 73 9.0 -70.5
32 6194 32 46 74 9.0 -69.0
33 6195 33 46 75 9.0 -67.5
34 6196 34 46 76 9.0 -66.0
35 6197 35 46 77 9.0 -64.5
36 6198 36 46 78 9.0 -6

In [5]:
# Import necessary libraries
import numpy as np
import matplotlib.pyplot as plt  # For plotting and visualization
# data source from 
# Tuinenburg, O. A., et al. High-resolution global atmospheric moisture connections 
# from evaporation to precipitation. Earth System Science Data 12, 3177-3188, (2020).

# Load various datasets for tracking and cell information
considered_cells = np.load("/Net/Groups/BGI/scratch/fhuang/utrack/Dataset/Further data/considered_cells.npy")
select_cell = np.load("sa_x_cell.npy")  # Selected cell indices

# Load the ERA-Interim data matrix for 2001-2018
f = "/Net/Groups/BGI/scratch/fhuang/utrack/Dataset/Matrices/Land_cell_to_grid(Era_Int_2001_2018)/Era_Int_2001_2018_matrix_yr.npy"
Era_Int_2001_2018_cell = np.load(f)

# Load longitude and latitude data for all cells
cell_longitudes = np.load("/Net/Groups/BGI/scratch/fhuang/utrack/Dataset/Further data/cell_longitudes.npy") 
cell_latitudes = np.load("/Net/Groups/BGI/scratch/fhuang/utrack/Dataset/Further data/cell_latitudes.npy")

def get_weights(x_cell, data):
    """
    Calculate percentage weights for a specific cell across all years.
    
    Parameters:
    x_cell (int): Index of the target cell
    data (ndarray): Input data matrix with shape [years*months, cells]
    
    Returns:
    ndarray: Percentage weights matrix with shape [107 latitudes, 240 longitudes]
    """
    # Extract data for the specific cell across all time steps
    Considered_cell = data[:, [x_cell]] 
    
    # Reshape from temporal to spatial grid (107 latitudes × 240 longitudes)
    Reshaped_array_cell = Considered_cell.reshape(107, 240)
    
    # Create longitude and latitude grids for the entire spatial domain
    longitudes_cells = np.tile(cell_longitudes, (107, 1))          # Repeat longitudes for all latitudes
    latitudes_cells = np.transpose(np.tile(cell_latitudes, (240, 1)))  # Repeat latitudes for all longitudes
    
    # Calculate percentage weights (normalize to sum to 100%)
    values_yr_percent_cell = Reshaped_array_cell / np.sum(Reshaped_array_cell) * 100
    
    return values_yr_percent_cell

def get_lat_square(lat, inv):
    """
    Calculate latitude boundaries for a square window around a given latitude.
    Handles edge cases near the poles.
    
    Parameters:
    lat (int): Center latitude index
    inv (int): Window radius in grid cells
    
    Returns:
    tuple: (upper_bound, lower_bound) latitude indices
    """
    # Calculate upper boundary, ensuring it doesn't go below 0
    if lat - inv < 0:
        up = 0
    else:
        up = lat - inv
    
    # Calculate lower boundary, ensuring it doesn't exceed maximum latitude (107)
    if lat + inv > 107:
        down = 107
    else:
        down = lat + inv
    
    return up, down

def get_square_weights(x_cell, inv):
    """
    Extract a square window of weights around a specific cell, handling longitude wrapping.
    
    Parameters:
    x_cell (int): Index of the target cell
    inv (int): Window radius in grid cells
    
    Returns:
    ndarray: Square window of weights with shape [2*inv+1, 2*inv+1] (or adjusted for edges)
    """
    # Get the full weight matrix for the cell
    weights = get_weights(x_cell)
    
    # Get the latitude and longitude indices of the target cell
    lat, lon = considered_cells[:, x_cell]
    
    # Calculate latitude boundaries for the square window
    up, down = get_lat_square(lat=lat, inv=inv)
    
    # Handle longitude wrapping (global grid wraps around 360°)
    if lon - inv < 0:
        # Case: window extends past the left edge (0°)
        left = lon - inv + 240  # Wrap around to the right side
        # Combine weights from the right end and left end of the grid
        select_square_cell = np.hstack((weights[up:down, left:], weights[up:down, :lon+inv]))
    
    elif lon + inv > 240:
        # Case: window extends past the right edge (360°)
        right = inv - (240 - lon)  # Calculate how much to take from the left side
        # Combine weights from the left end and right beginning of the grid
        select_square_cell = np.hstack((weights[up:down, lon-inv:], weights[up:down, :right]))
    
    else:
        # Normal case: window is entirely within the grid boundaries
        select_square_cell = weights[up:down, lon-inv:lon+inv]
    
    return select_square_cell

In [8]:
# Import xarray for working with labeled multi-dimensional arrays
import xarray as xr

# Load selected cell indices
select_cell = np.load("sa_x_cell.npy")

# Load longitude and latitude data for grid cells
cell_longitudes = np.load("/Net/Groups/BGI/scratch/fhuang/utrack/Dataset/Further data/cell_longitudes.npy") 
cell_latitudes = np.load("/Net/Groups/BGI/scratch/fhuang/utrack/Dataset/Further data/cell_latitudes.npy")

# Load considered cells information (latitude and longitude indices)
considered_cells = np.load("/Net/Groups/BGI/scratch/fhuang/utrack/Dataset/Further data/considered_cells.npy")

# Get number of samples from selected cells
nsample = select_cell.shape[0]

# Define window size for neighborhood extraction (10 cells in each direction)
inv = 10

# Initialize output data array for moisture contribution weights
# Dimensions: [samples, months(12), window_height(21), window_width(21), features(1)]
y_data = np.full((nsample, 12, inv*2+1, inv*2+1, 1), np.nan)

# List of months for processing
month_list = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", 
              "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

# Calculate actual longitude and latitude values from grid indices
# Adjust from grid coordinates to actual geographic coordinates
lon_values = (cell_longitudes + 0.75) - 180  # Convert to -180 to 180 range
lat_values = cell_latitudes + 0.75           # Adjust latitude values

# Load mask data (e.g., land/ocean mask) for filtering
mask = xr.open_dataarray('/Net/Groups/BGI/scratch/fhuang/utrack/uta_res/mask.nc')

# Loop through each month
for midx, month in enumerate(month_list):
    # Commented out: Alternative approach to load monthly data directly
    # filename = "/Net/Groups/BGI/scratch/fhuang/utrack/Dataset/Matrices/Land_cell_to_grid(Era_Int_2001_2018)/Era_Int_2001_2018_matrix_" + month + ".npy"
    # data = np.load(filename)
    
    # Loop through each sample/cell
    for i in range(nsample):
        # Get current cell index
        x_cell = select_cell[i]
        
        # Get latitude and longitude indices for the current cell
        lati, loni = considered_cells[:, x_cell]
        
        # Convert to 0-based indexing
        lati = lati - 1
        loni = loni - 120
        
        # Calculate boundaries for the neighborhood window
        up = lati - inv      # Top boundary
        down = lati + inv + 1  # Bottom boundary
        left = loni - inv    # Left boundary
        right = loni + inv + 1  # Right boundary
        
        # Print progress information for debugging
        print(i, x_cell, up, down, left, right)
        
        # Commented out: Alternative approach using get_weights function
        # w = get_weights(x_cell, data)[up:down, left:right]
        
        # Get moisture contribution weights for the current cell and month
        # This function returns masked and normalized weights
        weights_pic = get_upwind_weights_mask(x_cell, lat_values, lon_values, mask, period=str(month))
        
        # Extract the neighborhood window from the weights data
        y_data[i, midx, :, :, 0] = weights_pic[up:down, left:right]

# Save the processed moisture contribution data to file
np.save('/Net/Groups/BGI/scratch/fhuang/sm_uta_cnn/inv-10/y2.npy', y_data)

0 5920 33 54 57 78
1 5921 33 54 69 90
2 5922 33 54 70 91
3 6004 34 55 61 82
4 6005 34 55 62 83
5 6006 34 55 63 84
6 6007 34 55 64 85
7 6008 34 55 65 86
8 6009 34 55 66 87
9 6010 34 55 67 88
10 6011 34 55 68 89
11 6012 34 55 69 90
12 6013 34 55 70 91
13 6094 35 56 59 80
14 6095 35 56 60 81
15 6096 35 56 61 82
16 6097 35 56 62 83
17 6098 35 56 63 84
18 6099 35 56 64 85
19 6100 35 56 65 86
20 6101 35 56 66 87
21 6102 35 56 67 88
22 6103 35 56 68 89
23 6104 35 56 69 90
24 6105 35 56 70 91
25 6187 36 57 57 78
26 6188 36 57 58 79
27 6189 36 57 59 80
28 6190 36 57 60 81
29 6191 36 57 61 82
30 6192 36 57 62 83
31 6193 36 57 63 84
32 6194 36 57 64 85
33 6195 36 57 65 86
34 6196 36 57 66 87
35 6197 36 57 67 88
36 6198 36 57 68 89
37 6199 36 57 69 90
38 6200 36 57 70 91
39 6282 37 58 57 78
40 6283 37 58 58 79
41 6284 37 58 59 80
42 6285 37 58 60 81
43 6286 37 58 61 82
44 6287 37 58 62 83
45 6288 37 58 63 84
46 6289 37 58 64 85
47 6290 37 58 65 86
48 6291 37 58 66 87
49 6292 37 58 67 88
50 6293 37